In [2]:
import time
import json
import random

import numpy as np
import pynvml
import torch

from transformers import BartForConditionalGeneration, BartTokenizer
from datasets import load_dataset
from evaluate import load

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Initializing Model & Tokenizer

In [5]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)

# Loading Dataset

Selecting only 5000 data points from the test set. This needs to be done because processing more data points leads to GPU memory error on our system

In [7]:
dataset = load_dataset("ccdv/arxiv-summarization", "document")
test_data = dataset["test"].select(range(5000))

# Loading Evaluation Metrics

In [9]:
rouge = load("rouge")
bleu = load("bleu")
meteor = load("meteor")
bertscore = load("bertscore")

[nltk_data] Downloading package wordnet to /home/shivi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/shivi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/shivi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# Generating & Evaluating Summaries

In [11]:
def generate_summaries_and_evaluate(articles, references, batch_size=1):
    summaries = []
    inference_times = []
    memory_usages = []
    
    # Initialize GPU memory tracking
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    # Metric storage
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    rougeLSum_scores = []
    bleu_scores = []
    meteor_scores = []
    bert_precision_scores = []
    bert_recall_scores = []
    bert_f1_scores = []

    for i in range(0, len(articles), batch_size):
        batch_articles = articles[i:i + batch_size]
        batch_references = references[i:i + batch_size]

        # Track inference start time
        start_time = time.time()

        # Track GPU memory before processing
        mem_info_before = pynvml.nvmlDeviceGetMemoryInfo(handle)
        mem_used_before = mem_info_before.used

        # Tokenization & Model Generation
        inputs = tokenizer(
            ["summarize: " + article for article in batch_articles],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
        ).to(device)

        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=300,
            num_beams=2,
            early_stopping=True,
        )
        
        # Track inference end time
        end_time = time.time()
        inference_time = end_time - start_time
        inference_times.append(inference_time)

        # Track GPU memory after processing
        mem_info_after = pynvml.nvmlDeviceGetMemoryInfo(handle)
        mem_used_after = mem_info_after.used

        # Compute memory difference
        memory_usage = (mem_used_after - mem_used_before) / (1024**2)  # Convert to MB
        memory_usages.append(memory_usage)

        # Decode summaries
        batch_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(batch_summaries)

        # Compute Metrics for Each Summary
        rouge_result = rouge.compute(predictions=batch_summaries, references=batch_references)
        bleu_result = bleu.compute(predictions=batch_summaries, references=[[ref] for ref in batch_references])
        meteor_result = meteor.compute(predictions=batch_summaries, references=batch_references)
        bertscore_result = bertscore.compute(predictions=batch_summaries, references=batch_references, model_type="bert-base-uncased")

        # Store individual scores
        rouge1_scores.append(rouge_result["rouge1"])
        rouge2_scores.append(rouge_result["rouge2"])
        rougeL_scores.append(rouge_result["rougeL"])
        rougeLSum_scores.append(rouge_result["rougeLsum"])
        bleu_scores.append(bleu_result["bleu"])
        meteor_scores.append(meteor_result["meteor"])
        bert_precision_scores.append(np.mean(bertscore_result["precision"]))
        bert_recall_scores.append(np.mean(bertscore_result["recall"]))
        bert_f1_scores.append(np.mean(bertscore_result["f1"]))
        
        # Free CUDA memory after every batch
        torch.cuda.empty_cache()

    # Release GPU tracking resources
    pynvml.nvmlShutdown()

    # Compute Final Mean Scores
    final_scores = {
        "ROUGE-1": np.mean(rouge1_scores),
        "ROUGE-2": np.mean(rouge2_scores),
        "ROUGE-L": np.mean(rougeL_scores),
        "ROUGE-Lsum": np.mean(rougeLSum_scores),
        "BLEU": np.mean(bleu_scores),
        "METEOR": np.mean(meteor_scores),
        "BERTScore-Precision": np.mean(bert_precision_scores),
        "BERTScore-Recall": np.mean(bert_recall_scores),
        "BERTScore-F1": np.mean(bert_f1_scores)
    }

    return summaries, inference_times, memory_usages, final_scores

In [12]:
articles = test_data["article"]
references = test_data["abstract"]

torch.cuda.empty_cache()

summaries, inference_times, memory_usages, final_scores = generate_summaries_and_evaluate(articles, references)

In [13]:
avg_time_per_summary = sum(inference_times) / len(inference_times)
avg_memory_per_batch = sum(memory_usages) / len(memory_usages)

print(f"Avg Inference Time per Summary: {avg_time_per_summary:.4f} seconds")
print(f"Avg GPU Memory Usage per Batch: {avg_memory_per_batch:.2f} MB")
print(f"Metrics Scores: {final_scores}")

Avg Inference Time per Summary: 0.8377 seconds
Avg GPU Memory Usage per Batch: 150.73 MB
Metrics Scores: {'ROUGE-1': np.float64(0.25746444712686034), 'ROUGE-2': np.float64(0.06911944817806341), 'ROUGE-L': np.float64(0.1581346482099961), 'ROUGE-Lsum': np.float64(0.20590536863836306), 'BLEU': np.float64(0.01636518639995159), 'METEOR': np.float64(0.1437666564936916), 'BERTScore-Precision': np.float64(0.6101359673559665), 'BERTScore-Recall': np.float64(0.5264879395723343), 'BERTScore-F1': np.float64(0.5640772131025791)}


# Storing 100 Article, Reference & Generated Summaries for Human Evaluation

In [15]:
num_samples = min(100, len(articles))
sample_indices = random.sample(range(len(articles)), num_samples)

data = [
    {
        "article": articles[i],
        "reference_summary": references[i],
        "generated_summary": summaries[i]
    }
    for i in sample_indices
]

json_file = "bart-arxiv-summaries.json"
with open(json_file, "w", encoding="utf-8") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

print(f"✅ Saved {num_samples} randomly selected summaries to {json_file}")

✅ Saved 100 randomly selected summaries to bart-arxiv-summaries.json
